# Programming for AI, Assignment 3
NumPy, Pandas and Matplotlib on real measurement data

National University of Computer and Emerging Sciences | Fall 2026 | Total Marks: 100

## Instructions
- Work only in this notebook. Fill the cells marked `# YOUR CODE HERE` and the markdown cells marked **ANSWER**.
- Allowed libraries: numpy, pandas, matplotlib. Do not change the names of the data files.
- Use a fixed seed (`np.random.seed(42)`) wherever you need random numbers.
- Every interpretation must refer to actual numbers from your own output. Generic statements get zero.
- Your notebook must run top to bottom with Kernel > Restart and Run All. Notebooks that fail halfway lose 10 marks.
- Q1 and Q2 must be answered by reasoning first. You may run code afterwards only to check yourself, but your written answer must explain the reasoning.
- Submit through GitHub as described in the PDF.

| Question | Topic | Marks |
|---|---|---|
| 1 | Code tracing | 12 |
| 2 | Debugging | 10 |
| 3 | Sensor cube (NumPy) | 18 |
| 4 | Weather stations (Pandas) | 20 |
| 5 | Student performance (Pandas + NumPy) | 15 |
| 6 | Dashboard (Matplotlib) | 15 |
| 7 | Surfaces and animation | 10 |

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec
from matplotlib.animation import FuncAnimation, PillowWriter
np.random.seed(42)
print(np.__version__, pd.__version__)

## Question 1: Code tracing (12 marks, 3 each)
Predict the exact printed output of each block **without running it**. Write shapes as tuples and mention dtype where it matters. Then add one or two sentences saying which line is the trap.

### 1.a
```python
import numpy as np
a = np.arange(24).reshape(2, 3, 4)
b = a[:, ::-1, -2:]
b[0, 0, 0] = 99
print(b.shape)
print(a[0, 2, 2])
print(a[:, 1, :].sum(axis=0))
print(b.T.shape)
```
ANSWER:

Output:

(2, 3, 2)

99

[20 22 24 26]

(2, 3, 2)

Reasoning:

a[:, ::-1, -2:] keeps both days, reverses the 3 rows and keeps the last 2 columns, so b.shape is (2, 3, 2). Slicing returns a view, so writing b[0,0,0] = 99 changes a. b[0,0,0] is a[0, 2, 2] (row 0 of b is the last row of a, and column -2 is index 2), so a[0,2,2] prints 99. Row 1 of each day is untouched: [4,5,6,7] + [16,17,18,19] = [20 22 24 26]. b.T reverses the axes, so (2,3,2) becomes (2,3,2); the shape is palindromic so it looks unchanged.

Trap:

The view: the write goes through to a.

### 1.b
```python
import numpy as np
x = np.array([10, 20, 30, 40, 50, 60])
idx = np.array([-1, 0, -2, 2])
y = x[idx]
y[0] = 0
print(x)
z = x[1:4]
z[:] = 7
print(x)
print(x[[-1, -1, 0]])
```
ANSWER:

Output:

[10 20 30 40 50 60]

[10 7 7 7 50 60]

[60 60 10]

Reasoning:

Integer-array (fancy) indexing returns a copy, so y[0] = 0 does not change x. x[1:4] is a basic slice, which is a view, so z[:] = 7 overwrites positions 1 to 3 of x. Negative indices count from the end, so x[[-1,-1,0]] is [60 60 10].

Trap:

Fancy indexing copies, slicing does not.

### 1.c
```python
import numpy as np
s = np.array([3.9, -3.9, 2.5, 3.5, -0.5])
print(s.astype(int))
print(np.round(s))
print(np.array(['1', '2.5']).astype(float).dtype)
print((s > 0).sum(), (s > 0).any(), (np.abs(s) > 3).all())
```
ANSWER:

Output:

[3 -3 2 3 0]

[4. -4. 2. 4. -0.]

float64

3 True False

Reasoning:

astype(int) truncates toward zero (3.9 becomes 3, -3.9 becomes -3, -0.5 becomes 0). np.round rounds half to even: 2.5 becomes 2.0, 3.5 becomes 4.0, and -0.5 becomes -0. (negative zero). The strings 1 and 2.5 convert to float64. The positives are 3.9, 2.5 and 3.5, so the count is 3 and any is True; the absolute value of 2.5 is not above 3, so all is False.

Trap:

np.round(2.5) is 2, not 3, and astype(int) does not round.

### 1.d
```python
import numpy as np
A = np.arange(6).reshape(2, 3)
B = np.array([[10], [20]])
C = np.array([1, 2, 3])
print(A + B + C)
print(A @ C)
print(A * C)
print(np.cumprod(C + 1))
print(A.sum(axis=0) - A.sum(axis=1)[0])
```
ANSWER:

Output:

[[11 13 15]

[24 26 28]]

[8 26]

[[0 2 6]

[3 8 15]]

[2 6 24]

[0 2 4]

Reasoning:

A is [[0,1,2],[3,4,5]]. B is (2,1) and C is (3,), so A+B+C broadcasts to (2,3): row 0 gets 10 added, row 1 gets 20 added, then [1,2,3] is added to each row. A @ C is a matrix-vector product: [0+2+6, 3+8+15] = [8 26]. A times C with the star operator is element-wise, not a matrix product. C+1 is [2,3,4], so cumprod gives [2 6 24]. The column sums of A are [3 5 7] and A.sum(axis=1)[0] is 3, giving [0 2 4].

Trap:

The star operator is element-wise broadcasting, the @ operator is the matrix product.

## Question 2: Debugging (10 marks, 2 each)
For each snippet: (i) name the exact exception or the silent logical bug, (ii) explain why in one or two sentences, (iii) write corrected code in the code cell below it, (iv) state the corrected output or shape.

### 2.a
```python
A = np.array([[1, 2, 3], [4, 5, 6]])
B = np.array([[1, 2, 3], [4, 5, 6]])
print(np.dot(A, B))
```
Intended: the 2x2 matrix of dot products between rows of A and rows of B.

ANSWER:

Error:

ValueError: shapes (2,3) and (2,3) not aligned: 3 (dim 1) != 2 (dim 0).

Reason:

np.dot on 2-D arrays is a matrix product, which needs the inner dimensions to match (3 and 2). Rows-against-rows dot products need A @ B.T.

Expected output:

[[14 32]

[32 77]]

Shape is (2, 2).

In [ ]:
A = np.array([[1, 2, 3], [4, 5, 6]])
B = np.array([[1, 2, 3], [4, 5, 6]])
# np.dot(A, B) fails: inner dims 3 and 2 do not match. Rows-vs-rows needs A @ B.T
print(A @ B.T)

### 2.b
```python
scores = np.arange(12, dtype=float).reshape(3, 4)
centered = scores - scores.mean(axis=1)
print(centered)
```
Intended: subtract each row's own mean from that row.

ANSWER:

Error:

ValueError: operands could not be broadcast together with shapes (3,4) (3,).

Reason:

mean(axis=1) has shape (3,), and broadcasting aligns it with the last axis (length 4), so it fails. Use keepdims=True to get shape (3,1).

Expected output:

Every row is [-1.5 -0.5 0.5 1.5], shape (3, 4).

In [ ]:
scores = np.arange(12, dtype=float).reshape(3, 4)
# keepdims=True keeps the mean as (3, 1) so it broadcasts down each row
centered = scores - scores.mean(axis=1, keepdims=True)
print(centered)

### 2.c
```python
fig, ax = plt.subplots(2, 2, figsize=(8, 6))
ax[1].plot([1, 2, 3], [3, 1, 2])
ax[0, 0].set_title('Left')
ax[2, 0].hist(np.random.randn(100))
```
Intended: line plot in the bottom-left panel and histogram in the bottom-right panel.

ANSWER:

Error:

Two errors. ax[1] raises AttributeError: 'numpy.ndarray' object has no attribute 'plot'. ax[2, 0] raises IndexError: index 2 is out of bounds for axis 0 with size 2.

Reason:

ax[1] is a whole row of two axes (a NumPy array), not one axes. The grid is 2x2, so the rows are only 0 and 1. With a 2x2 grid always index ax[row, col].

Expected output:

Line plot in ax[1,0] (bottom-left), histogram in ax[1,1] (bottom-right), title Left on the top-left panel.

In [ ]:
fig, ax = plt.subplots(2, 2, figsize=(8, 6))
ax[1, 0].plot([1, 2, 3], [3, 1, 2])            # bottom-left
ax[0, 0].set_title('Left')
ax[1, 1].hist(np.random.randn(100))            # bottom-right
plt.close(fig)
print(ax.shape)

### 2.d
```python
marks = np.array([[35., 80., 55.], [20., 90., 45.]])
first = marks[:, 0]
first[first < 40] = 40          # curve for display only
print(marks.mean(axis=0))
```
Intended: the original marks must remain unchanged, the curved copy is used only for display.

ANSWER:

Error:

Silent logical bug, no exception is raised.

Reason:

marks[:, 0] is a basic slice, so first is a view. The boolean assignment writes through to marks, so the printed mean is [40. 85. 50.] instead of the true [27.5 85. 50.]. Fix: work on an explicit copy (marks.copy()) and curve the copy.

Expected output:

Original mean: [27.5 85. 50.]

Curved (display) mean: [40. 85. 50.]

In [ ]:
marks = np.array([[35., 80., 55.], [20., 90., 45.]])
curved = marks.copy()                           # independent copy for display
curved[:, 0] = np.where(curved[:, 0] < 40, 40, curved[:, 0])
print(marks.mean(axis=0))                       # original, unchanged
print(curved.mean(axis=0))                      # curved, display only

### 2.e
```python
x = np.linspace(0, 1, 50)
y = np.linspace(0, 1, 30)
X, Y = np.meshgrid(x, y)
Z = np.sin(3 * x) * np.cos(2 * Y)
fig, ax = plt.subplots()
im = ax.pcolormesh(X, Y, Z)
plt.colorbar()
```
Intended: heatmap of sin(3x)cos(2y) with a working colorbar and no shape error.

ANSWER:

Error:

RuntimeError: No mappable was found to use for colorbar creation.

Reason:

plt.colorbar() is given nothing to describe. Fix: fig.colorbar(im, ax=ax). Second issue: np.sin(3 times x) uses the 1-D x instead of the grid X. It only avoids a shape error because (50,) broadcasts against (30, 50); using X makes the intent explicit and safe.

Expected output:

X, Y and Z all have shape (30, 50) and the colorbar works.

In [ ]:
x = np.linspace(0, 1, 50)
y = np.linspace(0, 1, 30)
X, Y = np.meshgrid(x, y)
Z = np.sin(3 * X) * np.cos(2 * Y)               # use the 2-D grids X and Y
fig, ax = plt.subplots()
im = ax.pcolormesh(X, Y, Z)
fig.colorbar(im, ax=ax)                         # colorbar needs the mappable
plt.close(fig)
print(X.shape, Y.shape, Z.shape)

## Question 3: Sensor cube (18 marks)
`data/sensor_cube.npy` holds temperature readings from 6 sensors over 10 days, 24 hours per day. Axes are (day, hour, sensor). `data/sensor_meta.npz` holds `offsets` (calibration per sensor), `sensor_ids` and `days`. The logger writes -999 when a reading fails. Some other readings are physically impossible for this site (below 0 or above 60 degrees C).

**3a (3 marks)** Load both files. Print the shape, dtype and size of the cube, list the arrays stored in the npz archive and print `sensor_ids`. Report how many readings equal -999 and how many other readings are impossible. Why is -999 more dangerous than NaN if you take a mean directly?

In [ ]:
cube = np.load('data/sensor_cube.npy')
meta = np.load('data/sensor_meta.npz')
print('shape', cube.shape, '| dtype', cube.dtype, '| size', cube.size)
print('arrays in npz:', meta.files)
print('sensor_ids:', meta['sensor_ids'])
failed = (cube == -999)
impossible = ((cube < 0) | (cube > 60)) & ~failed      # exclude the -999 ones so nothing is counted twice
print('failed (-999):', failed.sum(), '| other impossible:', impossible.sum())
# -999 is a valid float, so np.mean silently includes it and returns a plausible-looking but wrong number.
# NaN makes the mean NaN, which is visible, and nanmean can then skip it on purpose.
print('naive mean with -999:', cube.mean().round(2), '| mean without bad values:', cube[~(failed | impossible)].mean().round(2))

**3b (4 marks)** Make a float copy of the cube (not a view) where every failed or impossible reading is NaN. Prove with `np.shares_memory` that your copy is independent. Then apply calibration by adding `offsets` using broadcasting only, no loops. State the shape of the broadcast operand and why it aligns with the last axis. Write the check that the original file array is untouched.

In [ ]:
original_check = np.load('data/sensor_cube.npy')     # fresh copy from disk to compare against later
clean = cube.astype(float)                           # astype copies by default
clean[failed | impossible] = np.nan
print('shares memory with original:', np.shares_memory(clean, cube))   # False -> independent

offsets = meta['offsets']                            # shape (6,)
# cube is (day, hour, sensor). A (6,) array is aligned from the RIGHT, so it lines up with the sensor axis;
# it is treated as (1, 1, 6) and stretched over day and hour.
print('offsets shape', offsets.shape, '-> broadcast as (1, 1, 6)')
cal = clean + offsets
print('original untouched:', np.array_equal(cube, original_check), '| NaN count in cal:', np.isnan(cal).sum())

**3c (4 marks)** Without loops compute: (i) the mean temperature of each sensor over all days and hours, (ii) the mean daily profile of each hour averaged over days and sensors (shape 24), (iii) the hottest (day, sensor) pair using daily means. NaN must not poison the result. State the axis you used for each and why.

In [ ]:
# (i) per sensor: collapse day (axis 0) and hour (axis 1) -> shape (6,)
sensor_mean = np.nanmean(cal, axis=(0, 1))
# (ii) per hour: collapse day (axis 0) and sensor (axis 2) -> shape (24,)
hourly_profile = np.nanmean(cal, axis=(0, 2))
# (iii) daily mean per (day, sensor): collapse hour (axis 1) -> shape (10, 6), then find the maximum ignoring NaN
daily = np.nanmean(cal, axis=1)
d, s = np.unravel_index(np.nanargmax(daily), daily.shape)
print('sensor means:', sensor_mean.round(2))
print('hourly profile:', hourly_profile.round(2))
print(f"hottest (day, sensor): day index {d} (day {meta['days'][d]}), sensor {meta['sensor_ids'][s]}, mean {daily[d, s]:.2f}")

**3d (4 marks)** For sensor T3 (index 2) compute the 3-hour moving average of the hourly series for day 4 (index 3) using `np.cumsum` and slicing only, not `np.convolve`. Fill NaN in that series with the mean of the series first. Report the length of the result and explain why it is not 24.

In [ ]:
series = cal[3, :, 2].copy()                          # day index 3, sensor T3 (index 2)
series[np.isnan(series)] = np.nanmean(series)        # fill NaN with the series mean
c = np.cumsum(np.insert(series, 0, 0.0))             # leading 0 so window sums are c[k+3] - c[k]
ma3 = (c[3:] - c[:-3]) / 3
print('length of moving average:', ma3.size)
# A window of 3 fits in 24 - 3 + 1 = 22 positions, so we lose window-1 = 2 points (no padding used).
print(ma3.round(2))

**3e (3 marks)** Count the hours per day in which at least one sensor is above 35 degrees C (use `any`), and the number of days in which all sensors stay below 35 for the whole day (use `all`). Using `np.unique` with counts on the hour of each day's peak (use the sensor-averaged series), state which hour is the most common peak. Interpret in two sentences: is the sensor network behaving like a daytime heating cycle?

In [ ]:
hot = cal > 35                                        # NaN > 35 is False, so NaN never counts as hot
hours_any = hot.any(axis=2).sum(axis=1)               # per day: hours where at least one sensor > 35
print('hours per day with any sensor > 35:', hours_any)
cool_days = np.all((cal < 35) | np.isnan(cal), axis=(1, 2)).sum()
print('days where every sensor stays below 35:', cool_days)
avg = np.nanmean(cal, axis=2)                         # sensor-averaged series (day, hour)
peak_hour = np.nanargmax(avg, axis=1)
vals, counts = np.unique(peak_hour, return_counts=True)
print('peak hours (value, count):', dict(zip(vals.tolist(), counts.tolist())))
print('most common peak hour:', vals[counts.argmax()])

ANSWER 3e interpretation:

Run the cell above and fill the XX values from your own output.

Most common peak hour: XX, which happened on XX of 10 days. Days with any sensor above 35: XX, for XX hours per day.

If the most common peak hour is in the early afternoon (about 12 to 16) and the hourly profile in 3c is lowest in the early morning, the network is behaving like a daytime heating cycle: temperature rises after sunrise, peaks after solar noon, then cools. If the peaks were spread over many different hours, something other than the sun would be driving the sensors.

## Question 4: Weather stations with Pandas (20 marks)
`data/weather_stations.csv` contains daily records for five cities. The file is messy: mixed date formats (`YYYY-MM-DD` and `DD/MM/YYYY`), temperatures stored as text such as `34.2C`, -999 as failed rain gauge value, missing humidity, pressure readings of 0 and duplicate rows.

**4a (6 marks)** Load and clean. Print `info()` before and after. Remove exact duplicates, parse both date formats into one datetime column (a plain `pd.to_datetime` call without care will silently misread days and months, check that), convert temperature to float, turn -999 into NaN, turn pressure 0 into NaN, fill missing humidity with the median humidity of the same city. Report the number of rows removed and the number of values you changed in each column. Name the cleaned frame `weather`.

In [ ]:
raw = pd.read_csv('data/weather_stations.csv')
print('--- info BEFORE ---'); raw.info()

def find(df, key):
    return next(c for c in df.columns if key in c.lower())
w = raw.rename(columns={find(raw, 'date'): 'date', find(raw, 'city'): 'city', find(raw, 'temp'): 'temp',
                        find(raw, 'humid'): 'humidity', find(raw, 'press'): 'pressure', find(raw, 'rain'): 'rain'})

n0 = len(w)
w = w.drop_duplicates().copy()                        # exact duplicate rows
print('rows removed as duplicates:', n0 - len(w))

# Dates: ISO strings are parsed as ISO, everything else as DD/MM/YYYY. A single to_datetime call would
# guess month-first for strings like 03/05/2025 and silently swap day and month.
d = w['date'].astype(str).str.strip()
iso = d.str.match(r'^\d{4}-\d{2}-\d{2}$')
parsed = pd.Series(pd.NaT, index=w.index, dtype='datetime64[ns]')
parsed[iso] = pd.to_datetime(d[iso], format='%Y-%m-%d')
parsed[~iso] = pd.to_datetime(d[~iso], format='%d/%m/%Y')
print('unparsed dates:', parsed.isna().sum())
w['date'] = parsed

temp_before = w['temp'].copy()
w['temp'] = pd.to_numeric(w['temp'].astype(str).str.replace('C', '', regex=False).str.strip(), errors='coerce')
print('temp values converted from text:', (temp_before.astype(str).str.contains('C')).sum(), '| NaN temps:', w['temp'].isna().sum())

n_rain = (w['rain'] == -999).sum();       w.loc[w['rain'] == -999, 'rain'] = np.nan
n_pres = (w['pressure'] == 0).sum();      w.loc[w['pressure'] == 0, 'pressure'] = np.nan
n_hum = w['humidity'].isna().sum()
w['humidity'] = w['humidity'].fillna(w.groupby('city')['humidity'].transform('median'))
print('changed -> rain -999 to NaN:', n_rain, '| pressure 0 to NaN:', n_pres, '| humidity filled:', n_hum)
weather = w.sort_values(['city', 'date']).reset_index(drop=True)
print('--- info AFTER ---'); weather.info()

**4b (5 marks)** Create a month column. Build a pivot table of mean temperature by city (rows) and month (columns), rounded to 1 decimal. Find the city with the largest rise from the first month to the last month. Add a column giving each row's temperature anomaly relative to its own city's mean, using `groupby` with `transform`.

In [ ]:
weather['month'] = weather['date'].dt.to_period('M')
pivot = weather.pivot_table(index='city', columns='month', values='temp', aggfunc='mean').round(1)
print(pivot)
rise = pivot.iloc[:, -1] - pivot.iloc[:, 0]
print('rise from first to last month:'); print(rise.round(1))
print('largest rise:', rise.idxmax(), round(rise.max(), 1))
weather['temp_anomaly'] = weather['temp'] - weather.groupby('city')['temp'].transform('mean')
print(weather[['city', 'date', 'temp', 'temp_anomaly']].head())

**4c (5 marks)** A heat spell is 3 or more consecutive days with temp above 38 in one city. Find every spell (city, start date, length). You may use NumPy on each city's sorted series, for example `diff` on a boolean array, but explain your logic in comments. If your data contains a missing calendar day for a city, say what you did about it.

In [ ]:
spells = []
full_range = pd.date_range(weather['date'].min(), weather['date'].max())
for city, g in weather.groupby('city'):
    s = g.groupby('date')['temp'].mean()              # one value per date (guards against repeated dates)
    missing = full_range.difference(s.index)
    print(f'{city}: {len(missing)} missing calendar day(s)', list(missing.strftime('%Y-%m-%d')))
    # Missing days are put back as NaN. NaN > 38 is False, so a missing day BREAKS a spell;
    # we do not assume the unseen day was hot.
    s = s.reindex(full_range)
    hot = (s.values > 38).astype(int)
    # Pad with 0 on both ends, then diff: +1 marks the first hot day of a run, -1 the day after its last.
    edges = np.diff(np.concatenate([[0], hot, [0]]))
    starts = np.where(edges == 1)[0]
    ends = np.where(edges == -1)[0]
    for a, b in zip(starts, ends):
        if b - a >= 3:
            spells.append((city, full_range[a].date(), int(b - a)))
spells_df = pd.DataFrame(spells, columns=['city', 'start', 'length'])
print(spells_df)

**4d (4 marks)** Compute the correlation matrix of temperature, humidity, pressure and rain. Then compute it separately for Karachi only. Interpret: which relationships are weak overall but strong inside one city, and why could pooling cities mislead you (name the effect)? Rain is mostly zero, so also state what that does to a correlation coefficient.

In [ ]:
cols = ['temp', 'humidity', 'pressure', 'rain']
corr_all = weather[cols].corr()
corr_khi = weather.loc[weather['city'] == 'Karachi', cols].corr()
print('ALL CITIES'); print(corr_all.round(2))
print('KARACHI ONLY'); print(corr_khi.round(2))
diff = (corr_khi - corr_all).abs().round(2)
print('absolute difference (Karachi vs pooled):'); print(diff)
print('share of rain days that are exactly 0:', round((weather['rain'] == 0).mean(), 3))
print('Spearman (rank) corr with rain, pooled:'); print(weather[cols].corr(method='spearman')['rain'].round(2))

ANSWER 4d interpretation:

Fill the XX values from the two matrices above.

Pairs that are weak pooled but strong in Karachi: XX (pooled r = XX, Karachi r = XX).

Pooling cities can mislead because of Simpson's paradox, also called confounding by group: cities differ in their average temperature, humidity and pressure, so the pooled correlation mixes differences between cities with day-to-day relationships inside a city. Opposite within-city trends can cancel out, or a pooled trend can appear that exists in no single city.

Rain is mostly zero (see the printed share of zero days), so the coefficient is driven by a handful of wet days: it is small in size, unstable, and one or two heavy days can change it. The Spearman line above is a more robust check.

## Question 5: Student performance (15 marks)
`data/student_performance.csv` has 240 students (some rows duplicated). Maximum marks are in the column names: quiz 20, assignment 20, midterm 25, final 35. Total = 100. Some midterms are the text `Absent`, some finals look like a typing slip (entered ten times too large), and some study hours are missing.

**5a (5 marks)** Clean the data. Absent counts as 0 for the total but must be kept in a separate boolean column. Fix finals that exceed 35 by the most defensible rule and say what rule you chose. Fill missing study hours with the section median. Create `total` and a `grade` column: A at 85 or more, B at 70 to 84.99, C at 55 to 69.99, D at 40 to 54.99, F below 40, using `np.where` (nested) or boolean masks, not a loop.

In [ ]:
raw_s = pd.read_csv('data/student_performance.csv')
def find(df, key):
    return next(c for c in df.columns if key in c.lower())
st = raw_s.rename(columns={find(raw_s, 'student'): 'student_id', find(raw_s, 'section'): 'section',
                           find(raw_s, 'quiz'): 'quiz', find(raw_s, 'assign'): 'assignment',
                           find(raw_s, 'mid'): 'midterm', find(raw_s, 'final'): 'final',
                           find(raw_s, 'study'): 'study_hours', find(raw_s, 'attend'): 'attendance'})
n0 = len(st)
st = st.drop_duplicates().copy()
print('duplicate rows removed:', n0 - len(st), '| remaining:', len(st))

st['midterm_absent'] = st['midterm'].astype(str).str.strip().str.lower().eq('absent')   # keep the flag
st['midterm'] = pd.to_numeric(st['midterm'], errors='coerce').fillna(0)                # Absent counts as 0
for c in ['quiz', 'assignment', 'final', 'study_hours', 'attendance']:
    st[c] = pd.to_numeric(st[c], errors='coerce')

# Rule for finals > 35: the problem says they were entered ten times too large, so divide by 10.
# That is more defensible than capping at 35 (which invents a mark) or dropping the row (loses the student).
# Any value that is still above 35 after dividing is capped at 35 as a safety net.
bad = st['final'] > 35
print('finals fixed by dividing by 10:', bad.sum())
st.loc[bad, 'final'] = st.loc[bad, 'final'] / 10
print('finals still above 35 (capped):', (st['final'] > 35).sum())
st['final'] = st['final'].clip(upper=35)

print('missing study hours filled:', st['study_hours'].isna().sum())
st['study_hours'] = st['study_hours'].fillna(st.groupby('section')['study_hours'].transform('median'))

st['total'] = st[['quiz', 'assignment', 'midterm', 'final']].sum(axis=1)
t = st['total']
st['grade'] = np.where(t >= 85, 'A', np.where(t >= 70, 'B', np.where(t >= 55, 'C', np.where(t >= 40, 'D', 'F'))))
students = st.reset_index(drop=True)
print(students['grade'].value_counts().sort_index())

**5b (5 marks)** Per section report: number of students, mean total, std of total, pass percentage (not F), and the student_id with the highest total. Then list students with attendance below 75 who still scored a B or better. Use a single boolean mask.

In [ ]:
g = students.groupby('section')
summary = pd.DataFrame({
    'n': g.size(),
    'mean_total': g['total'].mean().round(2),
    'std_total': g['total'].std().round(2),
    'pass_pct': g['grade'].apply(lambda s: (s != 'F').mean() * 100).round(1),
    'top_student': g.apply(lambda d: d.loc[d['total'].idxmax(), 'student_id'], include_groups=False),
})
print(summary)
mask = (students['attendance'] < 75) & students['grade'].isin(['A', 'B'])
print('low attendance (<75) but grade B or better:', mask.sum())
print(students.loc[mask, ['student_id', 'section', 'attendance', 'total', 'grade']])

**5c (5 marks)** Compute the correlation of study hours with total and with final. Fit a straight line of total on study hours with `np.polyfit` (degree 1) and report slope and intercept. Interpret the slope in plain words with units. Then compare mean total of absent-midterm students and everyone else and state whether this proves absence causes low marks. Mention at least one confounder visible in the data.

In [ ]:
r_total = students['study_hours'].corr(students['total'])
r_final = students['study_hours'].corr(students['final'])
print('corr(study_hours, total) =', round(r_total, 3), '| corr(study_hours, final) =', round(r_final, 3))
slope, intercept = np.polyfit(students['study_hours'], students['total'], 1)
print(f'total = {slope:.3f} * study_hours + {intercept:.2f}')
ab = students.loc[students['midterm_absent']]
rest = students.loc[~students['midterm_absent']]
print('mean total, absent midterm:', round(ab['total'].mean(), 2), '(n =', len(ab), ') | everyone else:', round(rest['total'].mean(), 2))
# Confounders visible in the data: do absent students also study less and attend less?
print('mean study hours  absent vs rest:', round(ab['study_hours'].mean(), 2), round(rest['study_hours'].mean(), 2))
print('mean attendance   absent vs rest:', round(ab['attendance'].mean(), 2), round(rest['attendance'].mean(), 2))
print('mean total WITHOUT the midterm component, absent vs rest:',
      round((ab['total'] - ab['midterm']).mean(), 2), round((rest['total'] - rest['midterm']).mean(), 2))

ANSWER 5c interpretation:

Fill the XX values from the output above.

Correlation of study hours with total: XX. With final: XX. Slope: XX marks per study hour. Intercept: XX.

The slope means that each extra study hour goes with about XX more marks out of 100 in the predicted total, and a student with zero study hours is predicted to score about XX.

Mean total of absent-midterm students: XX. Everyone else: XX.

This does not prove that absence causes low marks. Absent students automatically lose the midterm marks because they are counted as 0, which lowers their total by construction. The data also show other differences, such as attendance and study hours (quote the printed means), so those are confounders. The last printed line removes the midterm component to show the gap that remains.

## Question 6: Dashboard (15 marks)
Build one figure, `figsize=(14, 9)`, using `GridSpec` with 3 rows and 3 columns and constrained layout or `tight_layout`. Use your cleaned `weather`, `students` (from Q5) and the calibrated sensor cube. Required panels:
1. (top row, spans 2 columns) Daily mean temperature of each city as lines with different line styles, a shared legend outside the axes and a marked heat spell using `fill_between` or `axvspan`. Add a twin y-axis with mean humidity of one city (a different colour, with colour matched tick labels).
2. (top right) Horizontal bar chart of mean total per section with `errorbar` style whiskers showing std. Print the bar values as text.
3. (middle row, left) Histogram of student totals with bins every 5 marks from 0 to 100, a vertical dashed line at the mean, and `ax.annotate` with an arrow pointing to the tallest bin. Use data coordinates for the arrow target and axes coordinates for the text.
4. (middle row, middle) Scatter of study hours vs total coloured by attendance, with colorbar and the polyfit line from Q5c.
5. (middle right) Step plot of the hourly sensor-average profile (24 hours), y-axis limits chosen by you and justified in a comment.
6. (bottom row, spans all columns) `imshow` heatmap of the day by hour mean sensor temperature (10 x 24) with a diverging colormap centred on the overall mean using `vmin`/`vmax`, tick labels showing real day numbers, and an inset axes (`inset_axes` or `fig.add_axes`) showing the hour with the maximum anomaly as a small bar chart over days.
Mark split: layout 3, panels 1 to 2 4, panels 3 to 5 4, panel 6 3, labels/titles/legend quality 1. Save as PNG at 200 dpi and as PDF.

In [ ]:
# ---------- prepare inputs ----------
daily_temp = weather.pivot_table(index='date', columns='city', values='temp', aggfunc='mean')
hum_city = 'Karachi' if 'Karachi' in daily_temp.columns else daily_temp.columns[0]
daily_hum = weather[weather['city'] == hum_city].groupby('date')['humidity'].mean()
sec = students.groupby('section')['total'].agg(['mean', 'std'])
day_hour = np.nanmean(cal, axis=2)                    # (10, 24) day by hour
hourly = np.nanmean(cal, axis=(0, 2))                 # (24,)
overall = np.nanmean(cal)

fig = plt.figure(figsize=(14, 9), layout='constrained')
gs = GridSpec(3, 3, figure=fig)

# ---------- panel 1: temperature lines + heat spell + twin humidity axis ----------
ax1 = fig.add_subplot(gs[0, 0:2])
styles = ['-', '--', '-.', ':', (0, (5, 1))]
for (city, col), ls in zip(daily_temp.items(), styles):
    ax1.plot(daily_temp.index, col, ls=ls, lw=1.5, label=city)
if len(spells_df):
    sp = spells_df.sort_values('length', ascending=False).iloc[0]
    st0 = pd.Timestamp(sp['start'])
    ax1.axvspan(st0, st0 + pd.Timedelta(days=int(sp['length'])), color='orange', alpha=0.3,
                label=f"Heat spell ({sp['city']})")
ax1.set_ylabel('Daily mean temperature (°C)'); ax1.set_title('Daily mean temperature by city')
ax1.tick_params(axis='x', rotation=30)
ax1b = ax1.twinx()
hum_color = 'tab:olive'
ax1b.plot(daily_hum.index, daily_hum.values, color=hum_color, lw=1, alpha=0.8, label=f'Humidity, {hum_city} (right axis)')
ax1b.set_ylabel('Humidity (%)', color=hum_color); ax1b.tick_params(axis='y', colors=hum_color)
h1, l1 = ax1.get_legend_handles_labels(); h2, l2 = ax1b.get_legend_handles_labels()
fig.legend(h1 + h2, l1 + l2, loc='outside lower center', ncol=len(l1 + l2), frameon=False)   # shared legend outside the axes

# ---------- panel 2: horizontal bars with std whiskers ----------
ax2 = fig.add_subplot(gs[0, 2])
ypos = np.arange(len(sec))
ax2.barh(ypos, sec['mean'], xerr=sec['std'], capsize=4, color='steelblue', alpha=0.85)
for y, m in zip(ypos, sec['mean']):
    ax2.text(2, y, f'{m:.1f}', va='center', ha='left', color='white', fontweight='bold')
ax2.set_yticks(ypos, sec.index); ax2.set_xlabel('Mean total (whisker = 1 std)'); ax2.set_title('Total by section')

# ---------- panel 3: histogram, mean line, annotated tallest bin ----------
ax3 = fig.add_subplot(gs[1, 0])
bins = np.arange(0, 105, 5)
counts, edges, _ = ax3.hist(students['total'], bins=bins, color='seagreen', edgecolor='white')
m = students['total'].mean()
ax3.axvline(m, ls='--', color='k', label=f'mean {m:.1f}')
k = counts.argmax(); cx = (edges[k] + edges[k + 1]) / 2
ax3.annotate(f'tallest bin\n{edges[k]:.0f}-{edges[k+1]:.0f}: {int(counts[k])}',
             xy=(cx, counts[k]), xycoords='data',                 # arrow target in data coordinates
             xytext=(0.05, 0.85), textcoords='axes fraction',     # text placed in axes coordinates
             arrowprops=dict(arrowstyle='->'), va='top')
ax3.set_xlabel('Total marks'); ax3.set_ylabel('Students'); ax3.set_title('Distribution of totals'); ax3.legend(loc='center right')

# ---------- panel 4: scatter coloured by attendance + polyfit ----------
ax4 = fig.add_subplot(gs[1, 1])
sc = ax4.scatter(students['study_hours'], students['total'], c=students['attendance'], cmap='viridis', s=18)
fig.colorbar(sc, ax=ax4, label='Attendance (%)')
xs = np.linspace(students['study_hours'].min(), students['study_hours'].max(), 50)
ax4.plot(xs, slope * xs + intercept, 'r-', lw=2, label=f'fit: {slope:.2f}x + {intercept:.1f}')
ax4.set_xlabel('Study hours'); ax4.set_ylabel('Total'); ax4.set_title('Study hours vs total'); ax4.legend()

# ---------- panel 5: step plot of hourly profile ----------
ax5 = fig.add_subplot(gs[1, 2])
ax5.step(np.arange(24), hourly, where='mid', color='crimson')
# y-limits: 1 degree of padding around the data range, so the daily swing fills the panel without
# starting at zero (a zero baseline would flatten a ~15 degree cycle) and without clipping the extremes.
ax5.set_ylim(np.floor(hourly.min()) - 1, np.ceil(hourly.max()) + 1)
ax5.set_xlabel('Hour of day'); ax5.set_ylabel('Mean temp (°C)'); ax5.set_title('Hourly sensor profile'); ax5.set_xticks(range(0, 24, 4))

# ---------- panel 6: heatmap + inset ----------
ax6 = fig.add_subplot(gs[2, :])
dev = np.nanmax(np.abs(day_hour - overall))
im = ax6.imshow(day_hour, aspect='auto', cmap='RdBu_r', vmin=overall - dev, vmax=overall + dev)   # diverging, centred on the mean
ax6.set_yticks(range(day_hour.shape[0]), meta['days']); ax6.set_xticks(range(24))
ax6.set_xlabel('Hour'); ax6.set_ylabel('Day'); ax6.set_title('Mean sensor temperature, day by hour')
fig.colorbar(im, ax=ax6, label='°C (centre = overall mean)', pad=0.01)
hmax = int(np.nanargmax(hourly - overall))
ins = ax6.inset_axes([0.80, 0.08, 0.17, 0.52])
ins.bar(np.arange(day_hour.shape[0]), day_hour[:, hmax] - overall, color='k', alpha=0.8)
ins.set_title(f'Hour {hmax} anomaly by day', fontsize=8); ins.tick_params(labelsize=7)

fig.suptitle('Plant dashboard', fontsize=14)
fig.savefig('dashboard.png', dpi=200)
fig.savefig('dashboard.pdf')
plt.show()

ANSWER 6 (4 sentences):

Using specific numbers from your dashboard, tell a plant manager what the single most important finding is, and name one panel whose design could mislead a reader and why.

Your answer:

Most important finding: XX (quote your own numbers, for example the longest heat spell with its city, start date and length, and the hottest sensor day and hour).

A panel that can mislead is the section bar chart: the whiskers show one standard deviation, which is large compared with the gaps between the section means, so a small difference in means can look meaningful when it is well inside the spread. The twin-axis panel can also mislead because the two y-scales are chosen independently, which can make unrelated lines look related.

## Question 7: Surfaces and animation (10 marks)
Let f(x, y) = sin(r) / r with r = sqrt(x^2 + y^2) on x, y in [-10, 10].

**7a (4 marks)** Build the grid with `meshgrid` of 201 points per axis. Handle r = 0 correctly (the limit is 1; a plain division gives NaN, show you saw it and fixed it with `np.where`). Draw side by side: `pcolormesh` with a diverging colormap and symmetric `vmin`/`vmax`, and `contourf` with `contour` lines labelled using `clabel`. Add colorbars. Count (with Boolean sums) how many grid points have f below -0.1 and say what visual feature that corresponds to.

In [ ]:
x = np.linspace(-10, 10, 201)
X, Y = np.meshgrid(x, x)
R = np.sqrt(X**2 + Y**2)
with np.errstate(invalid='ignore', divide='ignore'):
    naive = np.sin(R) / R                              # 0/0 at the centre
print('NaN in naive version:', np.isnan(naive).sum(), 'at index', np.argwhere(np.isnan(naive)).tolist())
safe_R = np.where(R == 0, 1.0, R)                      # avoid dividing by zero
F = np.where(R == 0, 1.0, np.sin(R) / safe_R)          # limit of sin(r)/r at 0 is 1
print('NaN after fix:', np.isnan(F).sum(), '| F at centre:', F[100, 100])

fig, (a1, a2) = plt.subplots(1, 2, figsize=(12, 5), layout='constrained')
p = a1.pcolormesh(X, Y, F, cmap='RdBu_r', vmin=-1, vmax=1, shading='auto')   # symmetric limits so 0 is white
fig.colorbar(p, ax=a1); a1.set_title('pcolormesh'); a1.set_aspect('equal')
cf = a2.contourf(X, Y, F, levels=20, cmap='RdBu_r', vmin=-1, vmax=1)
cl = a2.contour(X, Y, F, levels=10, colors='k', linewidths=0.5)
a2.clabel(cl, inline=True, fontsize=7)
fig.colorbar(cf, ax=a2); a2.set_title('contourf + contour'); a2.set_aspect('equal')
plt.show()
print('grid points with f < -0.1:', (F < -0.1).sum(), 'of', F.size)
# These points form the concentric negative rings (troughs) around the central peak.

**7b (3 marks)** Make a 3D surface with `plot_surface` using `rstride=5, cstride=5`, then a `plot_wireframe` of the same data, and add a contour projection onto the floor (`zdir="z"`, with offset below the minimum). Use `view_init(elev=35, azim=-60)`. Explain what changes if rstride and cstride are set to 1 and why.

In [ ]:
fig = plt.figure(figsize=(14, 6))
for i, (title, kind) in enumerate([('plot_surface', 'surf'), ('plot_wireframe', 'wire')]):
    ax = fig.add_subplot(1, 2, i + 1, projection='3d')
    if kind == 'surf':
        ax.plot_surface(X, Y, F, rstride=5, cstride=5, cmap='viridis')
    else:
        ax.plot_wireframe(X, Y, F, rstride=5, cstride=5, color='k', linewidth=0.5)
    ax.contour(X, Y, F, zdir='z', offset=F.min() - 0.3, cmap='viridis')   # projection on the floor
    ax.set_zlim(F.min() - 0.3, F.max())
    ax.view_init(elev=35, azim=-60)
    ax.set_title(title)
plt.show()
# With rstride=cstride=1 every one of the 200 x 200 = 40,000 grid cells becomes its own polygon:
# rendering is much slower and heavier, edges crowd together (the wireframe turns almost solid black),
# and the picture gets no more informative because the function is already smooth at stride 5 (~40 x 40 patches).

**7c (3 marks)** Animate the travelling wave sin(r - 0.3 t)/r along the line y = 0 using `FuncAnimation` with 60 frames, `interval=50` and `blit=True`. Your update function must return the artist(s). Save as `wave.gif` with `PillowWriter`. Do not use `plt.show()` blocking calls. Say in a comment why blit needs the update function to return a tuple.

In [ ]:
xs = np.linspace(-10, 10, 401)
def wave(t):
    r = np.abs(xs)
    rr = np.where(r == 0, 1.0, r)
    return np.where(r == 0, 1.0, np.sin(r - 0.3 * t) / rr)

fig, ax = plt.subplots(figsize=(7, 4))
line, = ax.plot(xs, wave(0), lw=2)
ax.set_ylim(-0.5, 1.1); ax.set_xlabel('x (y = 0)'); ax.set_title('sin(r - 0.3 t) / r')

def update(frame):
    line.set_ydata(wave(frame))
    # blit redraws only the artists it receives back and iterates over the return value,
    # so it must be a sequence (a tuple) of artists, even when there is just one.
    return (line,)

ani = FuncAnimation(fig, update, frames=60, interval=50, blit=True)
ani.save('wave.gif', writer=PillowWriter(fps=20))
plt.close(fig)
print('saved wave.gif')

## Final checklist
- Restart and Run All works
- All ANSWER cells filled
- `dashboard.png`, `dashboard.pdf` and `wave.gif` are created by your notebook
- Code pushed to GitHub as described in the PDF